In [ ]:
import pandas as pd

#Q1:

roll_number = input("Enter your college roll number: ")
# Take last two digits
last_two_digits = roll_number[-2:]
categories = ["billing", "account", "general"]
# Fixed entries
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]
# Personalized entries
personalized_entries = []
for digit in last_two_digits:
    d = int(digit)
    category = categories[d % 3]
    if category == "billing":
        entry = {
            "question": "how can i check my payment status",
            "answer": "You can check your payment status from the Billing section.",
            "keywords": "payment status transaction bill",
            "category": "billing"
        }
    elif category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Go to Account Settings and update your registered mobile number.",
            "keywords": "mobile number update account",
            "category": "account"
        }
    else:
        entry = {
            "question": "how can i contact customer support",
            "answer": "You can contact customer support through the Help section.",
            "keywords": "support help contact assistance",
            "category": "general"
        }

    personalized_entries.append(entry)
# Combine all entries
all_entries = fixed_entries + personalized_entries
# Create DataFrame
df = pd.DataFrame(all_entries)
print(df.to_string(index=False))

# Q2:

def score_query(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        # Count matching words
        keyword_matches = query_words.intersection(keyword_words)
        question_matches = query_words.intersection(question_words)

        # Keywords have higher importance
        score = (len(keyword_matches) * 2) + len(question_matches)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    # Sort by confidence score
    results.sort(key=lambda x: x["score"], reverse=True)

    return results


query = input("\nEnter your FAQ query: ")

results = score_query(query, df)

print("\n========== Q2: MATCHING RESULTS ==========")

if results:
    for result in results:
        print("Question:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Confidence Score:", result["score"])
        print("-" * 40)
else:
    print("No matching FAQ found.")


# Q3: same_category()

def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]


# Take category of first personalized entry
personalized_category = personalized_entries[0]["category"]

print("\n========== Q3: SAME CATEGORY ==========")
print("Category selected:", personalized_category)

category_result = same_category(personalized_category, df)

print(category_result.to_string(index=False))


# Q4: Add New Keyword and Save CSV

print("\n========== Q4: ADD NEW KEYWORD ==========")

# Pick first entry
entry_index = 0

print("Selected FAQ:")
print(df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword: ")

# Add new keyword
df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

print("\nUpdated Entry:")
print(df.loc[entry_index].to_string())

# Save CSV
filename = roll_number + "_faq_data.csv"
df.to_csv(filename, index=False)

print("\nUpdated DataFrame saved as:", filename)


# Q5: Count FAQ entries per category

category_counts = df.groupby("category").size()

print(category_counts)


# Q6: Modified Scoring Function with Ties

def score_query_with_tie(query, df):

    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        keyword_matches = query_words.intersection(keyword_words)
        question_matches = query_words.intersection(question_words)

        score = (len(keyword_matches) * 2) + len(question_matches)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matching entries found.")
        return

    # Sort according to score
    results.sort(key=lambda x: x["score"], reverse=True)

    highest_score = results[0]["score"]

    # Get ALL entries having highest score
    best_matches = [
        result for result in results
        if result["score"] == highest_score
    ]

    print("\nHighest Confidence Score:", highest_score)

    if len(best_matches) > 1:
        print("TIE DETECTED!:\n")

        for result in best_matches:
            print("Question:", result["question"])
            print("Answer:", result["answer"])
            print("Category:", result["category"])
            print("Score:", result["score"])
            print("-" * 40)

    else:
        print("Best Match:")
        print("Question:", best_matches[0]["question"])
        print("Answer:", best_matches[0]["answer"])
        print("Category:", best_matches[0]["category"])
        print("Score:", best_matches[0]["score"])


# Q6 Demonstration


print("\nQuery 1: Produces a tie")
score_query_with_tie("fee", df)

print("\nQuery 2: Does not produce a tie")
score_query_with_tie("password", df)